## Synthetic data generation pipeline

Separately generate questions for RQ1, RQ2, RQ3.

**Reference**: 
> Simone Filice, Guy Horowitz, David Carmel, Zohar Karnin, Liane Lewin-Eytan, and Yoelle Maarek. 2025. Generating Diverse Q&A Benchmarks for RAG Evaluation with DataMorgana. doi:10.48550/arXiv.2501.12789 arXiv:2501.12789 [cs]

In [15]:
import requests
import random
import time
from typing import List, Dict, Any
import json
import uuid
from tqdm import tqdm

JSONType = Dict[str, Any]

## Below: bulk generate synthetic QAs with DataMorgana tool

ai71 API traditional method, which samples 1 category value for each category, according to the probabilities specified.

In [ ]:
import json
import time
from typing import Dict, List
import requests

from categorization_configs import *
from EXTRA_categorization_configs import *

BASE_URL = "https://api.ai71.ai/v1/"

API_KEY = "..." # TODO: enter your ai71 API key here.

In [ ]:
OUTPUT_DIR = "./CONF_QAs/"

In [2]:
def bulk_generate(n_questions: int, question_categorizations: List[Dict], user_categorizations: List[Dict]):
    resp = requests.post(
        f"{BASE_URL}bulk_generation",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "n_questions": n_questions,
            "question_categorizations": question_categorizations,
            "user_categorizations": user_categorizations
        }
    )
    resp.raise_for_status()
    request_id = resp.json()["request_id"]
    print(f"Request ID: {request_id}")
    
    result = wait_for_generation_to_finish(request_id)
    return result

def wait_for_generation_to_finish(request_id: str):
    while True:
        resp = requests.get(
            f"{BASE_URL}fetch_generation_results",
            headers={"Authorization": f"Bearer {API_KEY}"},
            params={"request_id": request_id},
        )
        resp.raise_for_status()
        if resp.json()["status"] == "completed":
            print("Generation completed!")
            return resp.json()
        else:
            print("Waiting for generation to finish...")
            time.sleep(2)

def save_results(results: Dict, output_filepath: str):
    """Save results to JSONL file (one JSON object per line)"""
    with open(output_filepath, "w", encoding="utf-8") as fout:
        # Assuming results contain a list of QA pairs
        if "results" in results:
            for qa_pair in results["results"]:
                fout.write(json.dumps(qa_pair, ensure_ascii=False) + "\n")
        else:
            # Fallback: save entire response
            fout.write(json.dumps(results, ensure_ascii=False) + "\n")
    print(f"Saved to {output_filepath}")

In [ ]:
# ============================================================================
# RQ1: GRANULARITY (3,000 questions)
# ============================================================================

print("=" * 80)
print("GENERATING RQ1: GRANULARITY")
print("=" * 80)

OUTPUT_DIR = "./CONF_QAs/"

# Question Complexity - Coarse (333 questions)
print("\n[RQ1] Question Complexity - Coarse (333 questions)")
results = bulk_generate(
    n_questions=333,
    question_categorizations=[question_complexity_coarse],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq1_complexity_coarse.jsonl")


In [ ]:
# ... continue generation for RQ1

# Question Complexity - Medium (333 questions)
print("\n[RQ1] Question Complexity - Medium (333 questions)")
results = bulk_generate(
    n_questions=333,
    question_categorizations=[question_complexity_medium],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq1_complexity_medium.jsonl")

# Question Complexity - Fine (334 questions)
print("\n[RQ1] Question Complexity - Fine (334 questions)")
results = bulk_generate(
    n_questions=334, # Over 6 cats.
    question_categorizations=[question_complexity_fine],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq1_complexity_fine.jsonl")

# Answer Type - Coarse (333 questions)
print("\n[RQ1] Answer Type - Coarse (333 questions)")
results = bulk_generate(
    n_questions=333,
    question_categorizations=[question_answertype_coarse],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq1_answertype_coarse.jsonl")

# Answer Type - Medium (333 questions)
print("\n[RQ1] Answer Type - Medium (333 questions)")
results = bulk_generate(
    n_questions=333,
    question_categorizations=[question_answertype_medium],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq1_answertype_medium.jsonl")

# Answer Type - Fine (334 questions)
print("\n[RQ1] Answer Type - Fine (334 questions)")
results = bulk_generate( # Over 7 cats
    n_questions=334,
    question_categorizations=[question_answertype_fine],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq1_answertype_fine.jsonl")

# Linguistic Variation - Coarse (333 questions)
print("\n[RQ1] Linguistic Variation - Coarse (333 questions)")
results = bulk_generate(
    n_questions=333,
    question_categorizations=[question_linguisticvariation_coarse],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq1_linguisticvar_coarse.jsonl")

# Linguistic Variation - Medium (333 questions)
print("\n[RQ1] Linguistic Variation - Medium (333 questions)")
results = bulk_generate(
    n_questions=333,
    question_categorizations=[question_linguisticvariation_medium],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq1_linguisticvar_medium.jsonl")

# Linguistic Variation - Fine (334 questions)
print("\n[RQ1] Linguistic Variation - Fine (334 questions)")
results = bulk_generate( # Over 6 cats.
    n_questions=334,
    question_categorizations=[question_linguisticvariation_fine],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq1_linguisticvar_fine.jsonl")


## (Continue RQ1) Below: bulk generate to fill out categories.
- for additional fine-grained categories.
- generating only the new categories, create separate configs with only the new categories at equal probabilities that sum to 1.0

In [ ]:
from EXTRA_categorization_configs import *
OUTPUT_DIR = "./CONF_QAGEN_ROOT/"

In [ ]:
# Complexity: 112 questions (2 new categories)
complexity_new = bulk_generate(
    n_questions=112,
    question_categorizations=[question_complexity_NEW_ONLY],
    user_categorizations=[user_expertise_categorization]
)
save_results(complexity_new, OUTPUT_DIR + "rq1c_NEW_complexity_fine.jsonl")

In [ ]:
# ============================================================================
# Generation calls
# ============================================================================


# 1.) question_complexity_NEW_ONLY: 
# --> before, had 55.6/category on avg (QAs over 6 categories = 334 total)
# --> now, generate 112 (to match 55.6/category)

# 2.) question_answertype_NEW_ONLY: 
# --> before, had 47.71/category on avg (QAs over 7 categories = 334 total)
# --> now generate 48 (to match 47.71/category)


# 3.) question_linguisticvariation_NEW_ONLY:
# --> before, had 55.6/category on avg (QAs over 6 categories = 334 total).
# --> now, generate 112 (to match 55.6/category)



# Answer Type: 48 questions (1 new category)
answertype_new = bulk_generate(
    n_questions=48,
    question_categorizations=[question_answertype_NEW_ONLY],
    user_categorizations=[user_expertise_categorization]
)
save_results(answertype_new, OUTPUT_DIR +"rq1f_NEW_answertype_fine.jsonl")

# Linguistic Variation: 112 questions (2 new categories)
linguisticvar_new = bulk_generate(
    n_questions=112,
    question_categorizations=[question_linguisticvariation_NEW_ONLY],
    user_categorizations=[user_expertise_categorization]
)
save_results(linguisticvar_new, OUTPUT_DIR +"rq1i_NEW_linguisticvar_fine.jsonl")


## RQ2 Below: Generate synthetic QAs for RQ2 (COMPLEMENTARITY) (2,000 questions)

In [ ]:
# ============================================================================
# RQ2: COMPLEMENTARITY (2,000 questions)
# ============================================================================

print("\n" + "=" * 80)
print("GENERATING RQ2: COMPLEMENTARITY")
print("=" * 80)

# Set 1: Complexity (400 questions)
print("\n[RQ2] Set 1: Complexity (400 questions)") ################## 2 cats.
results = bulk_generate(
    n_questions=400,
    question_categorizations=[rq2_set1_complexity],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq2_set1_complexity.jsonl")

# Set 2: Answer Type (400 questions)
print("\n[RQ2] Set 2: Answer Type (400 questions)") ################## 2 cats
results = bulk_generate(
    n_questions=400,
    question_categorizations=[rq2_set2_answertype],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq2_set2_answertype.jsonl")

# Set 3: Vocabulary (400 questions)
print("\n[RQ2] Set 3: Vocabulary (400 questions)") ################## 2 cats
results = bulk_generate(
    n_questions=400,
    question_categorizations=[rq2_set3_vocabulary],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq2_set3_vocabulary.jsonl")

# Set 4: Phrasing (400 questions)
print("\n[RQ2] Set 4: Phrasing (400 questions)") ################## NOTE --  4 cats here, but only 2 for above ones
results = bulk_generate(
    n_questions=400,
    question_categorizations=[rq2_set4_phrasing],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq2_set4_phrasing.jsonl")

# Set 5: User Expertise (400 questions)
# NOTE: This uses a simple factoid categorization with user expertise
print("\n[RQ2] Set 5: User Expertise (400 questions)") ################## 2 cats
question_factuality_simple = {
    "categorization_name": "question_factuality",
    "categories": [
        {
            "name": "factoid",
            "description": "A question seeking a specific, concise piece of information.",
            "probability": 0.5
        },
        {
            "name": "open_ended",
            "description": "A question inviting detailed or exploratory responses.",
            "probability": 0.5
        }
    ]
}
results = bulk_generate(
    n_questions=400,
    question_categorizations=[question_factuality_simple],
    user_categorizations=[user_expertise_categorization]
)
save_results(results, OUTPUT_DIR + "rq2_set5_expertise.jsonl")

print("\n" + "=" * 80)
print("ALL GENERATIONS COMPLETE!")
print("Total questions generated: 5,000")
print("=" * 80)

## Generate expample questions -- to demonstrate hierarchy:

Doc id: <urn:uuid:2fb61cd1-1c80-46be-9797-5be72d77ae79>

Categories:
1st QA: question_complexity = "simple" (coarse) + novice
- Q: What should patients do if they are currently taking losartan tablets that might be recalled?
- A: "Consumers should speak with their doctor to discuss the recall before they stop taking the drug, or if they have experienced any adverse effects that may be related to the drug."

2nd QA: question_complexity = "extractive_span" (fine) + novice
- Q: What should patients do before stopping their losartan medication after hearing about the recall?
- A: "Consumers should speak with their doctor to discuss the recall before they stop taking the drug."

________________________
3rd QA: linguistic_variation = "distant_from_document" (coarse) + novice
- Q: What should people do if they are currently taking blood pressure medication that might be contaminated?
- A: "Consumers should speak with their doctor to discuss the recall before they stop taking the drug, or if they have experienced any adverse effects that may be related to the drug."

4th QA: linguistic_variation = "abstraction_level_shift" (fine) + novice
- Q: If I'm taking losartan medication for high blood pressure, what should I do after hearing about a recall?
- A: Consumers should speak with their doctor to discuss the recall before they stop taking the drug, or if they have experienced any adverse effects that may be related to the drug. For questions about the recall, people can contact Camber Pharmaceuticals at 866-495-1995 from 9 a.m. to 5 p.m. ET on weekdays.

- Q: If I'm taking losartan medication for high blood pressure, what should I do after hearing about a recall?
- A: "Consumers should speak with their doctor to discuss the recall before they stop taking the drug, or if they have experienced any adverse effects that may be related to the drug. For questions about the recall, people can contact Camber Pharmaceuticals at 866-495-1995 from 9 a.m. to 5 p.m. ET on weekdays."

- Q: Is it safe to immediately stop taking losartan if it's part of the recall?
- "Consumers should not stop taking the drug without first speaking with their doctor to discuss the recall. This is recommended even if they have experienced adverse effects that may be related to the drug."